# DLsite 音声作品自动翻译

DLsite 日语音声作品 → 中日双语 SRT 字幕

**支持多音频文件，共用背景信息，各自独立翻译。**

**运行前确认**：菜单栏 → 修改 → 笔记本设置 → 硬件加速器选 T4 GPU

依次运行每个格子即可。

In [ ]:
# ① 克隆仓库 & 安装依赖
# 请将下方 URL 替换为你自己的仓库地址
!git clone https://github.com/your-username/DLsiteAudioTranslator.git
%cd DLsiteAudioTranslator
!pip install -r requirements.txt -q
!nvidia-smi

In [ ]:
# ② 导入模块 & 初始化
import sys
sys.path.insert(0, '/content/DLsiteAudioTranslator')

from pathlib import Path
from google.colab import files
from api_client import init_openai_client, usage_stats
from asr import run_asr
from background import AudioBackground
from proofread import run_smart_proofread
from translate import run_parallel_translation
from utils import format_srt_time

client = init_openai_client()

In [ ]:
# ③ 输入背景信息（所有音频共用）
# 包括：作品标题、故事背景、登场角色
background = AudioBackground.input_interactive()

In [ ]:
# ④ 上传音频文件（可多选）
print('📤 上传音频文件（mp3 / m4a / wav / mp4，可多选）:')
uploaded = files.upload()
audio_files = list(uploaded.keys())
print(f'✅ 共接收 {len(audio_files)} 个音频文件')

In [ ]:
# ⑤ 粘贴トラックリスト → 正则拆轨 → 对齐到文件名 → 确认
# 交互：粘贴整份リスト后单独一行输入 END
# 也可非交互：把列表赋给 TRACKLIST 后用 apply_tracklist
#
# TRACKLIST = """
# 1. タイトル [01:17]
# タグ
# 2. タイトル [36:52]
# タグ
# """
# track_descs = AudioBackground.apply_tracklist(TRACKLIST, audio_files)

track_descs = AudioBackground.input_track_descriptions(audio_files)
background.track_descriptions = track_descs


In [ ]:
# ⑥ 逐个处理：ASR → 校对 → 翻译 → 写 SRT

srt_files = []

for audio_path in audio_files:
    filename = Path(audio_path).name
    print(f"\n{'='*50}")
    print(f"🎬 处理: {filename}")
    print(f"{'='*50}")

    # Step 1: ASR
    raw_asr = run_asr(audio_path)
    if not raw_asr:
        print(f"⚠️ {filename} 未识别到任何内容，跳过")
        continue

    # Step 2: 校对（背景辅助）
    proofed_data = run_smart_proofread(client, raw_asr, background, filename)

    # Step 3: 翻译
    final_data = run_parallel_translation(client, proofed_data, background, filename)

    # Step 4: 写 SRT
    srt_file = f"{Path(audio_path).stem}_bilingual.srt"
    with open(srt_file, 'w', encoding='utf-8') as f:
        for i, s in enumerate(final_data, 1):
            f.write(
                f"{i}\n"
                f"{format_srt_time(s['start'])} --> {format_srt_time(s['end'])}\n"
                f"{s['ja']}\n{s['zh']}\n\n"
            )

    print(f"📄 已生成: {srt_file}")
    srt_files.append(srt_file)

# 汇总
print(f"\n{'='*50}")
print(f"✅ 全部完成！共生成 {len(srt_files)} 个 SRT 文件")
print(f"   Token 消耗估算: {usage_stats.total_tokens}")
print(f"{'='*50}")

# 逐个下载
for srt_file in srt_files:
    files.download(srt_file)